In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import re
from tabulate import tabulate


import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
breakfast = pd.read_json(path_or_buf='/content/breakfast.jsonl', lines=True)
dinner = pd.read_json(path_or_buf='/content/dinner.jsonl', lines=True)
lunch = pd.read_json(path_or_buf='/content/lunch.jsonl', lines=True)

data = pd.concat([breakfast, dinner, lunch]).drop_duplicates('id').reset_index(drop=True)

kbju_df = pd.DataFrame(data['kbju'].tolist())

data = pd.concat([data, kbju_df], axis=1)
data.head(3)

,id,url,page,title,description,dish_image,kbju,ready_in,kitchen_time,cuisine,breadcrumbs,category_path,common_allergens,ingredients,recipe,calories,protein,fat,carbs
0,269944,https://food.ru/recipes/269944-gorjachie-syrny...,1,Горячие сырные тосты,"Хлеб, начинка и немного теплоты — вот и весь р...",https://cdn.food.ru/unsigned/fit/640/480/ce/0/...,"{'calories': 234.32, 'protein': 10.11, 'fat': ...",15 минут,5 минут,европейская,"[Рецепты, Закуски, Закуски с хлебом или лаваше...",Рецепты > Закуски > Закуски с хлебом или лаваш...,"[Белок коровьего молока, Злаки, содержащие глю...","[Тостовый хлеб 4 ломтик = 160 г, Сыр 150 г, Тв...",### подготовиться\n![подготовиться](https://cd...,234.32,10.11,16.23,16.82
1,270281,https://food.ru/recipes/270281-bulochki-s-loso...,1,Булочки с лососем и яйцом,"Действуйте параллельно, чтобы сэкономить время...",https://cdn.food.ru/unsigned/fit/640/480/ce/0/...,"{'calories': 221.48, 'protein': 11.04, 'fat': ...",25 минут,15 минут,североамериканская,"[Рецепты, Закуски, Закуски с хлебом или лаваше...",Рецепты > Закуски > Закуски с хлебом или лаваш...,"[Белок коровьего молока, Злаки, содержащие глю...","[Филе форели 150 г, Булочка для бургера 2 шт. ...",### подготовиться\n![подготовиться](https://cd...,221.48,11.04,13.43,13.20
2,269552,https://food.ru/recipes/269552-sloiki-iz-testa...,1,Слойки из теста фило с клубникой,У этих слоек тонкое и хрустящее по краям тесто...,https://cdn.food.ru/unsigned/fit/640/480/ce/0/...,"{'calories': 303.9, 'protein': 3.93, 'fat': 18...",40 минут,20 минут,интернациональная,"[Рецепты, Выпечка, Выпечка из слоеного теста, ...",Рецепты > Выпечка > Выпечка из слоеного теста ...,"[Белок коровьего молока, Клубника, Орехи, Яйцо]","[Тесто фило 6 лист = 180 г, Сливочное масло 60...",![](https://cdn.food.ru/unsigned/fit/640/480/c...,303.90,3.93,18.97,26.11


### Время предобрабатываем

In [3]:
def time_to_minutes(time_str):
    """
    Конвертирует строку времени в минуты.
    Обрабатывает: '55 минут', '2 часа 30 минут', '1 час', '9 часов 10 минут'
    """
    if pd.isna(time_str):
        return np.nan

    time_str = str(time_str).lower().strip()

    # Инициализируем переменные
    hours = 0
    minutes = 0

    # Обрабатываем часы
    if 'час' in time_str:
        hour_part = time_str.split('час')[0].strip()
        hour_match = re.search(r'\d+', hour_part)
        if hour_match:
            hours = int(hour_match.group())

    # Обрабатываем минуты
    if 'минут' in time_str:
        if 'час' in time_str:
            minute_part = time_str.split('час')[1]
        else:
            minute_part = time_str

        # Извлекаем число минут
        minute_match = re.search(r'\d+', minute_part)
        if minute_match:
            minutes = int(minute_match.group())

    # Если нет ни часов, ни минут, но есть просто число (например "55")
    elif re.search(r'^\d+$', time_str):
        minutes = int(time_str)

    return hours * 60 + minutes

In [4]:
data['ready_in_minutes'] = data['ready_in'].apply(time_to_minutes)
data['kitchen_time_in_minutes'] = data['kitchen_time'].apply(time_to_minutes)

In [5]:
cuisine_dummies = pd.get_dummies(data['cuisine'], prefix='cuisine')
data = pd.concat([data, cuisine_dummies], axis=1)

bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)

### Объединяем разные кухни в группы

In [6]:
# создаём словарь групп
groups = {
    "asian": [
        "cuisine_азиатская", "cuisine_китайская", "cuisine_японская",
        "cuisine_тайская", "cuisine_индийская", "cuisine_бурятская",
        "cuisine_среднеазиатская", 'cuisine_восточноазиатская',
        'cuisine_вьетнамская', 'cuisine_корейская'
    ],

    "european": [
        "cuisine_австрийская", "cuisine_английская", "cuisine_британская",
        "cuisine_датская", "cuisine_европейская", "cuisine_испанская",
        "cuisine_итальянская", "cuisine_немецкая", "cuisine_норвежская",
        "cuisine_прибалтийская", "cuisine_скандинавская", "cuisine_финская",
        "cuisine_французская", "cuisine_чешская", "cuisine_шведская",
        "cuisine_швейцарская", "cuisine_греческая",
        'cuisine_средиземноморская', 'cuisine_венгерская', 'cuisine_болгарская',
        'cuisine_ирландская', 'cuisine_польская', 'cuisine_португальская',
        'cuisine_сербская', 'cuisine_сицилийская', 'cuisine_хорватская',
        'cuisine_шотландская'
    ],

    "eastern": [
        "cuisine_арабская", "cuisine_армянская", "cuisine_азербайджанская",
        "cuisine_балканская", "cuisine_ближневосточная", "cuisine_восточная",
        "cuisine_грузинская", "cuisine_дагестанская", "cuisine_еврейская",
        "cuisine_израильская", "cuisine_кабардино-балкарская",
        "cuisine_кавказская", "cuisine_турецкая", "cuisine_чеченская",
         'cuisine_восточноевропейская',
        'cuisine_иранская', 'cuisine_узбекская'
    ],

    "slavic": [
        "cuisine_русская", "cuisine_украинская",
        "cuisine_белорусская", "cuisine_славянская",
        "cuisine_советская", "cuisine_молдавская",
        "cuisine_татарская",
        'cuisine_латышская', 'cuisine_литовская',
    ],
    "america": [
        "cuisine_американская", "cuisine_североамериканская",
        'cuisine_интернациональная',
    ],
    'mexica': [ "cuisine_латиноамериканская", 'cuisine_гавайская',
        "cuisine_мексиканская", "cuisine_североамериканская",
        'cuisine_перуанская'
    ]
}

for group_name, columns in groups.items():
    data[group_name] = data[columns].max(axis=1)

all_old_cols = [col for cols in groups.values() for col in cols]
data = data.drop(columns=all_old_cols)
data = data.drop(columns=['cuisine_авторская', 'cuisine_африканская', 'cuisine_марокканская', 'cuisine_монгольская'])

In [7]:
product_dict = {
    "рыба": ["рыба", 'рыбны', "форел", "лосос", "треск"],
    "морепродукты": ["креветк", "кальмар", "мидии", 'осминог', 'гребешк', 'морепродукт'],
    "свинина": ["свинин", "свиной"],
    "говядина": ["говядин", "говяжий"],
    'курица': ['куриц', 'кура ', 'курин'],
    "сыр": ["сыр", "моцарелл", "пармезан"],
    "картофель": ["картофел", "картошк"],
    "лук": [" лук"],
    "чеснок": ["чеснок", 'чесночн'],
    "помидоры": ["помидор", "томат"],
    "печень": ["печень"],
    "молоко": ["молоко"],
    "творог": ["творог"],
    "оливки": ["оливк", "маслин"],
    "сельдерей": ["сельдере"],
    "кинза": ["кинза", 'кинзы'],
    'тыква': ['тыкв'],
    'баклажан': ['баклажан'],
    'орехи': ['орех', 'кешью', 'фундук', 'миндал', 'грецки', 'макадами', 'арахис', 'фисташк']
}

In [8]:
for key, words in product_dict.items():
    data[key] = data["ingredients"].apply(
        lambda ing_list: int(
            any(word in " ".join(ing_list).lower() for word in words)
        )
    )

data["category_lvl1"] = data["breadcrumbs"].apply(
    lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None
)

In [9]:
data.category_lvl1.value_counts()

,count
category_lvl1,
Вторые блюда,1366
Выпечка,274
Десерты,252
Закуски,247
Первые блюда,232
Салаты,211
Гарниры,70
Напитки,21
Ингредиенты,18


In [10]:
data = data[~data.category_lvl1.isin(['Соусы и маринады', 'Заготовки', 'Партнерские проекты', 'От редакции Food.ru', 'Напитки'])]
data.shape

(2689, 47)

### Аллергены

In [11]:
allergens = [
    'Белок коровьего молока',
    'Злаки, содержащие глютен',
    'Яйцо',
    'Рыба',
    'Сельдерей',
    'Соя',
    'Пищевые добавки',
    'Горчица',
    'Орехи',
    'Клубника',
    'Кунжут',
    'Арахис',
    'Ракообразные',
    'Моллюски'
]

data['common_allergens'] = data['common_allergens'].apply(lambda x: x if isinstance(x, list) else [])

# Создаём бинарные колонки по фиксированному списку
for allergen in allergens:
    data[allergen] = data['common_allergens'].apply(lambda x: int(allergen in x))

# Добавляем колонку "Нет", когда ни одного аллергена нет
data['Нет'] = (data[allergens].sum(axis=1) == 0).astype(int)

data.head()

,id,url,page,title,description,dish_image,kbju,ready_in,kitchen_time,cuisine,breadcrumbs,category_path,common_allergens,ingredients,recipe,calories,protein,fat,carbs,ready_in_minutes,kitchen_time_in_minutes,asian,european,eastern,slavic,america,mexica,рыба,морепродукты,свинина,говядина,курица,сыр,картофель,лук,чеснок,помидоры,печень,молоко,творог,оливки,сельдерей,кинза,тыква,баклажан,орехи,category_lvl1,Белок коровьего молока,"Злаки, содержащие глютен",Яйцо,Рыба,Сельдерей,Соя,Пищевые добавки,Горчица,Орехи,Клубника,Кунжут,Арахис,Ракообразные,Моллюски,Нет
0,269944,https://food.ru/recipes/269944-gorjachie-syrny...,1,Горячие сырные тосты,"Хлеб, начинка и немного теплоты — вот и весь р...",https://cdn.food.ru/unsigned/fit/640/480/ce/0/...,"{'calories': 234.32, 'protein': 10.11, 'fat': ...",15 минут,5 минут,европейская,"[Рецепты, Закуски, Закуски с хлебом или лаваше...",Рецепты > Закуски > Закуски с хлебом или лаваш...,"[Белок коровьего молока, Злаки, содержащие глю...","[Тостовый хлеб 4 ломтик = 160 г, Сыр 150 г, Тв...",### подготовиться\n![подготовиться](https://cd...,234.32,10.11,16.23,16.82,15,5,0,1,0,0,0,0,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,Закуски,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,270281,https://food.ru/recipes/270281-bulochki-s-loso...,1,Булочки с лососем и яйцом,"Действуйте параллельно, чтобы сэкономить время...",https://cdn.food.ru/unsigned/fit/640/480/ce/0/...,"{'calories': 221.48, 'protein': 11.04, 'fat': ...",25 минут,15 минут,североамериканская,"[Рецепты, Закуски, Закуски с хлебом или лаваше...",Рецепты > Закуски > Закуски с хлебом или лаваш...,"[Белок коровьего молока, Злаки, содержащие глю...","[Филе форели 150 г, Булочка для бургера 2 шт. ...",### подготовиться\n![подготовиться](https://cd...,221.48,11.04,13.43,13.20,25,15,0,0,0,0,1,1,1,0,0,0,1,1,0,1,0,0,0,0,0,1,0,0,0,0,0,Закуски,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0
2,269552,https://food.ru/recipes/269552-sloiki-iz-testa...,1,Слойки из теста фило с клубникой,У этих слоек тонкое и хрустящее по краям тесто...,https://cdn.food.ru/unsigned/fit/640/480/ce/0/...,"{'calories': 303.9, 'protein': 3.93, 'fat': 18...",40 минут,20 минут,интернациональная,"[Рецепты, Выпечка, Выпечка из слоеного теста, ...",Рецепты > Выпечка > Выпечка из слоеного теста ...,"[Белок коровьего молока, Клубника, Орехи, Яйцо]","[Тесто фило 6 лист = 180 г, Сливочное масло 60...",![](https://cdn.food.ru/unsigned/fit/640/480/c...,303.90,3.93,18.97,26.11,40,20,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,Выпечка,1,0,1,0,0,0,0,0,1,1,0,0,0,0,0
3,269545,https://food.ru/recipes/269545-PEchenye-jablok...,1,Печеные яблоки под шубой,"«Шуба» получается сладкая, поэтому яблоки лучш...",https://cdn.food.ru/unsigned/fit/640/480/ce/0/...,"{'calories': 108.94, 'protein': 1.410000000000...",35 минут,20 минут,интернациональная,"[Рецепты, Десерты, Горячие десерты]",Рецепты > Десерты > Горячие десерты,[Яйцо],"[Яблоки 3 шт. = 450 г, Белок куриного яйца 2 ш...",![](https://cdn.food.ru/unsigned/fit/640/480/c...,108.94,1.41,0.30,26.41,35,20,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Десерты,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,270053,https://food.ru/recipes/270053-treugolnichki-s...,1,Треугольнички с моцареллой и окороком,Сэндвич с расплавленным сыром — простой и тако...,https://cdn.food.ru/unsigned/fit/640/480/ce/0/...,"{'calories': 233.09, 'protein': 12.07, 'fat': ...",20 минут,20 минут,итальянская,"[Рецепты, Закуски, Закуски с хлебом или лаваше...",Рецепты > Закуски > Закуски с хлебом или лаваш...,"[Белок коровьего молока, Злаки, содержащие глю...","[Свиной окорок 150 г, Тостовый хлеб 320 г, Моц...",### подготовиться\n![подготовиться](https://cd...,233.09,12.07,10.81,22.40,20,20,0,1,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,Закуски,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [12]:
data.to_csv('recipes.csv')
data.to_json('recipes.jsonl', orient='records', lines=True)

## Генерация юзеров для проверки работоспособности системы

In [13]:
np.random.seed(42)

def generate_users(n=1000):

    df = pd.DataFrame()

    # 1. БАЗОВАЯ ДЕМОГРАФИЯ

    df["user_id"] = range(1, n+1)

    # Пол (примерно 48% мужчины, 52% женщины)
    df["пол"] = np.random.choice(
        ["Мужской", "Женский"],
        size=n,
        p=[0.48, 0.52]
    )

    # Возраст (нормальное распределение)
    df["возраст"] = np.clip(
        np.random.normal(loc=38, scale=12, size=n),
        18, 70
    ).astype(int)

    # Рост по полу
    male_height = np.random.normal(178, 7, n)
    female_height = np.random.normal(165, 6, n)

    df["рост_см"] = np.where(
        df["пол"] == "Мужской",
        male_height,
        female_height
    ).astype(int)

    # Вес через BMI
    bmi = np.random.normal(25, 4, n)
    df["вес_кг"] = (bmi * (df["рост_см"]/100)**2).round(1)


    # 2. СТРАТЕГИЯ ПИТАНИЯ

    df["стратегия"] = np.random.choice(
        ["Похудение", "Поддержание массы", "Набор массы"],
        size=n,
        p=[0.55, 0.25, 0.20]
    )

    df["уровень_активности"] = np.random.choice(
        ["Низкий", "Умеренный", "Высокий"],
        size=n,
        p=[0.3, 0.5, 0.2]
    )


    # 3. ВРЕМЯ ГОТОВКИ

    df["активное_время_готовки_мин"] = np.clip(
        np.random.normal(35, 15, n),
        10, 120
    ).astype(int)

    df["пассивное_время_мин"] = np.clip(
        np.random.normal(50, 25, n),
        10, 240
    ).astype(int)

    # 4. ПРЕДПОЧТЕНИЯ КУХОНЬ (1–10)
    # СНГ -- высокий приоритет славянской и европейской

    cuisines = {
        "asian": 5,
        "european": 8,
        "eastern": 7,
        "slavic": 9,
        "america": 6,
        "mexica": 4
    }

    for cuisine, mean_val in cuisines.items():
        df[cuisine] = np.clip(
            np.random.normal(mean_val, 2, n),
            1, 10
        ).round().astype(int)

    # 5. ПРЕДПОЧТЕНИЯ ПРОДУКТОВ (1–10)
    # СНГ распределение

    products = {
        "рыба": 6,
        "морепродукты": 6,
        "свинина": 8,
        "говядина": 7,
        "курица": 9,
        "сыр": 8,
        "картофель": 9,
        "лук": 9,
        "чеснок": 8,
        "помидоры": 8,
        "печень": 5,
        "молоко": 5,
        "творог": 7,
        "оливки": 5,
        "сельдерей": 4,
        "кинза": 5,
        "тыква": 6,
        "баклажан": 7,
        "орехи": 7
    }

    for product, mean_val in products.items():
        df[product] = np.clip(
            np.random.normal(mean_val, 2, n),
            1, 10
        ).round().astype(int)

    # 6. АЛЛЕРГИИ
    # Реальные вероятности (редкие)

    allergies = [
        'Белок коровьего молока',
        'Злаки, содержащие глютен',
        'Яйцо',
        'Сельдерей',
        'Соя',
        'Пищевые добавки',
        'Горчица',
        'Клубника',
    ]

    allergy_probs = {
        'Белок коровьего молока': 0.05,
        'Злаки, содержащие глютен': 0.04,
        'Яйцо': 0.02,
        'Сельдерей': 0.01,
        'Соя': 0.02,
        'Пищевые добавки': 0.01,
        'Горчица': 0.01,
        'Клубника': 0.02,
    }

    has_any_allergy = np.zeros(n)

    peanut_base = np.random.binomial(1, 0.09, n)
    df['Орехи'] = peanut_base

    # Если есть арахис → высокая вероятность кунжута
    df['Кунжут'] = np.where(
        peanut_base == 1,
        np.random.binomial(1, 0.6, n),   # 60% совпадение
        np.random.binomial(1, 0.01, n)   # иначе редко
    )
    df['Арахис'] = np.where(
        peanut_base == 1,
        np.random.binomial(1, 0.9, n),
        np.random.binomial(1, 0.03, n)
    )

    seafood_base = np.random.binomial(1, 0.09, n)
    df['Рыба'] = seafood_base

    df['Ракообразные'] = np.where(
        seafood_base == 1,
        np.random.binomial(1, 0.7, n),
        np.random.binomial(1, 0.01, n)
    )

    df['Моллюски'] = np.where(
        seafood_base == 1,
        np.random.binomial(1, 0.65, n),
        np.random.binomial(1, 0.01, n)
    )

    for allergy in ['Орехи', 'Кунжут', 'Арахис', 'Рыба', 'Ракообразные', 'Моллюски']:
      has_any_allergy += df[allergy]

    for allergy in allergies:
        df[allergy] = np.random.binomial(1, allergy_probs[allergy], n)
        has_any_allergy += df[allergy]

    df["Нет"] = (has_any_allergy == 0).astype(int)

    return df



users_df = generate_users(5000)

users_df.head(10)


,user_id,пол,возраст,рост_см,вес_кг,стратегия,уровень_активности,активное_время_готовки_мин,пассивное_время_мин,asian,european,eastern,slavic,america,mexica,рыба,морепродукты,свинина,говядина,курица,сыр,картофель,лук,чеснок,помидоры,печень,молоко,творог,оливки,сельдерей,кинза,тыква,баклажан,орехи,Орехи,Кунжут,Арахис,Рыба,Ракообразные,Моллюски,Белок коровьего молока,"Злаки, содержащие глютен",Яйцо,Сельдерей,Соя,Пищевые добавки,Горчица,Клубника,Нет
0,1,Мужской,30,176,72.6,Похудение,Умеренный,10,59,3,10,7,8,9,5,6,3,5,8,8,10,8,10,10,9,5,3,4,5,6,4,5,7,6,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
1,2,Женский,18,161,55.7,Поддержание массы,Умеренный,26,55,3,5,7,10,3,9,4,4,5,6,9,5,10,7,10,10,8,4,10,9,7,4,7,7,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,3,Женский,33,159,63.2,Похудение,Низкий,25,63,6,6,5,8,10,5,6,8,10,5,10,5,10,10,7,8,4,4,9,5,6,5,9,6,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,4,Женский,48,167,77.5,Поддержание массы,Умеренный,35,62,6,8,4,8,8,3,7,10,8,7,9,8,9,10,6,8,4,5,8,4,5,4,5,2,5,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0
4,5,Мужской,44,176,68.4,Похудение,Низкий,34,44,6,5,7,10,3,1,9,7,8,3,9,7,8,6,7,8,4,3,6,6,4,2,5,6,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
5,6,Мужской,43,179,79.6,Набор массы,Высокий,37,43,4,6,5,7,5,3,8,8,6,9,6,9,10,10,9,8,3,1,9,3,7,4,6,5,6,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
6,7,Мужской,34,180,72.2,Поддержание массы,Низкий,37,25,7,9,6,10,3,3,6,7,8,7,6,5,6,10,7,5,4,3,4,6,1,4,8,9,4,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
7,8,Женский,22,168,72.9,Набор массы,Низкий,39,32,4,7,10,9,7,6,6,7,7,10,10,10,10,10,8,7,5,7,4,7,1,2,3,7,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
8,9,Женский,44,161,59.1,Похудение,Умеренный,20,48,3,10,7,10,9,3,8,7,10,10,10,8,10,7,9,8,1,5,5,4,8,6,8,6,5,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
9,10,Женский,27,167,61.5,Похудение,Высокий,24,54,8,5,9,9,6,2,6,7,6,8,7,8,10,6,10,10,3,4,6,5,4,3,5,9,6,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0


In [15]:
def calculate_bmr(row):
    if row['пол'] == 'Мужской':
        return 10*row['вес_кг'] + 6.25*row['рост_см'] - 5*row['возраст'] + 5
    else:
        return 10*row['вес_кг'] + 6.25*row['рост_см'] - 5*row['возраст'] - 161

users_df['bmr'] = users_df.apply(calculate_bmr, axis=1)

activity_map = {
    'Низкий': 1.2,
    'Умеренный': 1.55,
    'Высокий': 1.9
}

users_df['tdee'] = users_df['bmr'] * users_df['уровень_активности'].map(activity_map)

def adjust_goal(row):
    if row['стратегия'] == 'Похудение':
        return row['tdee'] * 0.8
    elif row['стратегия'] == 'Набор массы':
        return row['tdee'] * 1.15
    else:
        return row['tdee']

users_df['target_calories'] = users_df.apply(adjust_goal, axis=1)
users_df = users_df.drop('tdee', axis=1)

In [16]:
users_df.to_json('users.jsonl', orient='records', lines=True)